# KKBox Revenue Leakage Analysis — Data Cleaning
### Reads from: Data/ (raw) | Writes to: data/cleaned/ (parquet)

In [1]:
import pandas as pd
import numpy as np
import os
import time
import warnings
warnings.filterwarnings('ignore')

RAW     = r"./data/raw"
CLEANED = r"./data/cleaned"
CHUNK   = 500_000

print(f"RAW     -> {RAW}")
print(f"CLEANED -> {CLEANED}")


RAW     -> ./data/raw
CLEANED -> ./data/cleaned


## 1. Clean transactions_v2.csv

**Decisions:**
- Convert dates from int → datetime
- Fix dtypes: payment_method_id → int8, plan_price/amount_paid → int16, flags → int8
- Add `plan_type`: categorize plan_duration_days
- Add `is_discounted`: amount_paid < plan_price
- Add `is_payment_failed`: amount_paid == 0 AND is_cancel == 0
- Add `is_free_plan`: plan_price == 0
- `is_overpaid` (0.16%) — too small, not added
- Same user_id + transaction_date duplicates: 61,234 rows (4.28%) — KEEP (valid business events)
- No rows dropped

In [2]:
start = time.time()

chunks = []
for chunk in pd.read_csv(f"{RAW}/transactions_v2.csv", chunksize=CHUNK):
    chunks.append(chunk)
txn = pd.concat(chunks, ignore_index=True)
rows_before = len(txn)

# Convert dates
txn['transaction_date']       = pd.to_datetime(txn['transaction_date'].astype(str),       format='%Y%m%d', errors='coerce')
txn['membership_expire_date'] = pd.to_datetime(txn['membership_expire_date'].astype(str), format='%Y%m%d', errors='coerce')

# plan_type
def map_plan_type(days):
    if days == 30:    return '30-day'
    elif days == 90:  return '90-day'
    elif days == 180: return '180-day'
    elif days == 365: return '365-day'
    else:             return 'other'

txn['plan_type']         = txn['payment_plan_days'].apply(map_plan_type).astype('category')
txn['is_discounted']     = (txn['actual_amount_paid'] < txn['plan_list_price']).astype('int8')
txn['is_payment_failed'] = ((txn['actual_amount_paid'] == 0) & (txn['is_cancel'] == 0)).astype('int8')
txn['is_free_plan']      = (txn['plan_list_price'] == 0).astype('int8')

# Fix dtypes
txn['payment_method_id']  = txn['payment_method_id'].astype('int8')
txn['payment_plan_days']  = txn['payment_plan_days'].astype('int16')
txn['plan_list_price']    = txn['plan_list_price'].astype('int16')
txn['actual_amount_paid'] = txn['actual_amount_paid'].astype('int16')
txn['is_auto_renew']      = txn['is_auto_renew'].astype('int8')
txn['is_cancel']          = txn['is_cancel'].astype('int8')

# Rename
txn = txn.rename(columns={
    'msno':                   'user_id',
    'payment_plan_days':      'plan_duration_days',
    'plan_list_price':        'plan_price',
    'actual_amount_paid':     'amount_paid',
    'membership_expire_date': 'expire_date',
})

# Duplicate check: same user_id + transaction_date
dup_count = txn.duplicated(subset=['user_id', 'transaction_date'], keep=False).sum()
print(f"Same user_id + transaction_date: {dup_count:,} rows ({dup_count/len(txn)*100:.2f}%) — kept as valid business events")

# Save
txn.to_parquet(f"{CLEANED}/transactions_clean.parquet", index=False)

print(f"\ntransactions_clean.parquet saved")
print(f"  Rows    : {rows_before:,} -> {len(txn):,}")
print(f"  Columns : {list(txn.columns)}")
print(f"\n  Dtypes:")
print(txn.dtypes.to_string())
print(f"\n  Flag summary:")
print(f"    is_payment_failed : {txn['is_payment_failed'].sum():,}")
print(f"    is_discounted     : {txn['is_discounted'].sum():,}")
print(f"    is_free_plan      : {txn['is_free_plan'].sum():,}")
print(f"\n  Time: {time.time()-start:.1f}s")
print(f"\n  Sample (3 rows):")
display(txn[['user_id','transaction_date','plan_type','plan_price','amount_paid',
             'is_discounted','is_payment_failed','is_free_plan']].head(3))


Same user_id + transaction_date: 61,234 rows (4.28%) — kept as valid business events



transactions_clean.parquet saved
  Rows    : 1,431,009 -> 1,431,009
  Columns : ['user_id', 'payment_method_id', 'plan_duration_days', 'plan_price', 'amount_paid', 'is_auto_renew', 'transaction_date', 'expire_date', 'is_cancel', 'plan_type', 'is_discounted', 'is_payment_failed', 'is_free_plan']

  Dtypes:
user_id                          str
payment_method_id               int8
plan_duration_days             int16
plan_price                     int16
amount_paid                    int16
is_auto_renew                   int8
transaction_date      datetime64[us]
expire_date           datetime64[us]
is_cancel                       int8
plan_type                   category
is_discounted                   int8
is_payment_failed               int8
is_free_plan                    int8

  Flag summary:
    is_payment_failed : 21,362
    is_discounted     : 9,639
    is_free_plan      : 18,413

  Time: 2.6s

  Sample (3 rows):


,user_id,transaction_date,plan_type,plan_price,amount_paid,is_discounted,is_payment_failed,is_free_plan
0,++6eU4LsQ3UQ20ILS7d99XK8WbiVgbyYL4FUgzZR134=,2017-01-31,90-day,298,298,0,0,0
1,++lvGPJOinuin/8esghpnqdljm6NXS8m8Zwchc7gOeA=,2015-08-09,30-day,149,149,0,0,0
2,+/GXNtXWQVfKrEDqYAzcSw2xSPYMKWNj22m+5XkVQZc=,2017-03-03,30-day,180,180,0,0,0


## 2. Clean members_v3.csv

**Decisions:**
- DROP `bd` (67% invalid — not salvageable)
- KEEP `gender` but mark as not-to-use in analysis (65% null)
- DROP row where `registered_via == -1` (1 row)
- Convert `registration_init_time` → datetime, rename to `registration_date`
- Fix dtypes: city → int8, registered_via → int8

In [3]:
start = time.time()

members = pd.read_csv(f"{RAW}/members_v3.csv")
rows_before = len(members)

# Drop bd — 67% invalid, not salvageable
members = members.drop(columns=['bd'])

# Drop registered_via == -1 (1 bad row)
members = members[members['registered_via'] != -1]

# Convert date
members['registration_init_time'] = pd.to_datetime(
    members['registration_init_time'].astype(str), format='%Y%m%d', errors='coerce'
)

# Fix dtypes — city and registered_via are category codes, not numeric
members['city']           = members['city'].astype('category')
members['registered_via'] = members['registered_via'].astype('category')

# Rename
members = members.rename(columns={
    'msno':                   'user_id',
    'city':                   'city',
    'gender':                 'gender',
    'registered_via':         'registration_channel',
    'registration_init_time': 'registration_date',
})

members.to_parquet(f"{CLEANED}/members_clean.parquet", index=False)

print(f"members_clean.parquet saved")
print(f"  Rows    : {rows_before:,} -> {len(members):,}  (dropped {rows_before - len(members):,})")
print(f"  Columns : {list(members.columns)}")
print(f"\n  Dtypes:")
print(members.dtypes.to_string())
print(f"\n  city categories       : {sorted(members['city'].cat.categories.tolist())}")
print(f"  reg_channel categories: {sorted(members['registration_channel'].cat.categories.tolist())}")
print(f"\n  Sample (3 rows):")
display(members.head(3))


members_clean.parquet saved
  Rows    : 6,769,473 -> 6,769,472  (dropped 1)
  Columns : ['user_id', 'city', 'gender', 'registration_channel', 'registration_date']

  Dtypes:
user_id                            str
city                          category
gender                             str
registration_channel          category
registration_date       datetime64[us]

  city categories       : [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
  reg_channel categories: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 16, 17, 18, 19]

  Sample (3 rows):


,user_id,city,gender,registration_channel,registration_date
0,Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=,1,NaN,11,2011-09-11
1,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,NaN,7,2011-09-14
2,cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=,1,NaN,11,2011-09-15


## 3. Aggregate user_logs_v2.csv → user_engagement

**Decisions:**
- Parse `date` from int → datetime
- Aggregate to user level: avg_daily_songs, total_listen_secs, active_days, engagement_score
- Keep total_secs outliers (extreme users are real behaviour)
- Fix dtypes: active_days → int8, scores → float32

In [4]:
start = time.time()

agg_chunks = []
total_rows = 0

for chunk in pd.read_csv(f"{RAW}/user_logs_v2.csv", chunksize=CHUNK):
    total_rows += len(chunk)

    chunk['date'] = pd.to_datetime(chunk['date'].astype(str), format='%Y%m%d', errors='coerce')

    num_unq_safe = chunk['num_unq'].replace(0, np.nan)
    chunk['engagement_score'] = (
        chunk['num_100'] * 1.0 +
        chunk['num_985'] * 0.8 +
        chunk['num_75']  * 0.6 +
        chunk['num_50']  * 0.4 +
        chunk['num_25']  * 0.2
    ) / num_unq_safe

    chunk['total_plays'] = chunk[['num_25','num_50','num_75','num_985','num_100']].sum(axis=1)
    agg_chunks.append(chunk)

logs = pd.concat(agg_chunks, ignore_index=True)

user_engagement = (
    logs.groupby('msno')
    .agg(
        avg_daily_songs   = ('total_plays',      'mean'),
        total_listen_secs = ('total_secs',       'sum'),
        active_days       = ('date',             'count'),
        engagement_score  = ('engagement_score', 'mean'),
    )
    .reset_index()
)

# Fill NaN engagement (caused by num_unq == 0 edge cases) with 0
user_engagement['engagement_score'] = user_engagement['engagement_score'].fillna(0)

# Fix dtypes
user_engagement['avg_daily_songs']   = user_engagement['avg_daily_songs'].astype('float32')
user_engagement['total_listen_secs'] = user_engagement['total_listen_secs'].astype('float32')
user_engagement['active_days']       = user_engagement['active_days'].astype('int8')
user_engagement['engagement_score']  = user_engagement['engagement_score'].astype('float32')

# Rename
user_engagement = user_engagement.rename(columns={'msno': 'user_id'})

user_engagement.to_parquet(f"{CLEANED}/user_engagement.parquet", index=False)

print(f"user_engagement.parquet saved")
print(f"  Total rows processed : {total_rows:,}")
print(f"  Unique users         : {len(user_engagement):,}")
print(f"  Columns              : {list(user_engagement.columns)}")
print(f"\n  Dtypes:")
print(user_engagement.dtypes.to_string())
print(f"\n  Null check:")
print(user_engagement.isnull().sum().to_string())
print(f"\n  Time: {time.time()-start:.1f}s")
print(f"\n  Sample (3 rows):")
display(user_engagement.head(3))


user_engagement.parquet saved
  Total rows processed : 18,396,362
  Unique users         : 1,103,894
  Columns              : ['user_id', 'avg_daily_songs', 'total_listen_secs', 'active_days', 'engagement_score']

  Dtypes:
user_id                  str
avg_daily_songs      float32
total_listen_secs    float32
active_days             int8
engagement_score     float32

  Null check:
user_id              0
avg_daily_songs      0
total_listen_secs    0
active_days          0
engagement_score     0

  Time: 26.0s

  Sample (3 rows):


,user_id,avg_daily_songs,total_listen_secs,active_days,engagement_score
0,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,22.461538,117907.421875,26,1.173285
1,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,35.129032,192527.890625,31,0.910103
2,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,20.250000,115411.257812,28,1.048487


## 4. train_v2.csv — No cleaning needed, save as parquet

In [5]:
train = pd.read_csv(f"{RAW}/train_v2.csv")

# Fix dtype + rename
train['is_churn'] = train['is_churn'].astype('int8')
train = train.rename(columns={'msno': 'user_id'})

train.to_parquet(f"{CLEANED}/train_clean.parquet", index=False)

print(f"train_clean.parquet saved")
print(f"  Rows       : {len(train):,}")
print(f"  Columns    : {list(train.columns)}")
print(f"  Dtypes     :\n{train.dtypes.to_string()}")
print(f"  Churn rate : {train['is_churn'].mean()*100:.2f}%")


train_clean.parquet saved
  Rows       : 970,960
  Columns    : ['user_id', 'is_churn']
  Dtypes     :
user_id      str
is_churn    int8
  Churn rate : 8.99%


## 5. Cleaning Summary

In [6]:
import os

files = [
    ("transactions_clean.parquet", "transactions_v2.csv"),
    ("members_clean.parquet",      "members_v3.csv"),
    ("user_engagement.parquet",    "user_logs_v2.csv (aggregated)"),
    ("train_clean.parquet",        "train_v2.csv"),
]

print(f"{'File':<35} {'Size (MB)':>10}")
print("-" * 47)
for parquet, source in files:
    path = os.path.join(CLEANED, parquet)
    size = os.path.getsize(path) / (1024**2)
    print(f"  {parquet:<33} {size:>8.1f} MB   <-  {source}")

print(f"\nAll cleaned files saved to: {CLEANED}")
print(f"Raw files untouched in    : {RAW}")


File                                 Size (MB)
-----------------------------------------------
  transactions_clean.parquet            68.3 MB   <-  transactions_v2.csv
  members_clean.parquet                305.3 MB   <-  members_v3.csv
  user_engagement.parquet               58.4 MB   <-  user_logs_v2.csv (aggregated)
  train_clean.parquet                   42.3 MB   <-  train_v2.csv

All cleaned files saved to: ./data/cleaned
Raw files untouched in    : ./data/raw
